# Stage 3 — LLM Explanation Generation

**Reads:** `outputs/ontology_results.json`  
**Writes:** `outputs/explanations.json`

Generates a natural-language, user-adaptive explanation for each text using
the Qwen LLM, grounded in the ontology triples from Stage 2.

Each record saved = Stage 2 record + `"explanation": "..."`

> **Tip:** You can test a single explanation in Section 6 before running the
> full batch. The LLM is only loaded once.

## 1. Imports

In [1]:
from config import EXPLANATIONS_PATH, ONTOLOGY_RESULTS_PATH
from model_loaders import load_llm
from pipeline_helpers import (
    SYSTEM_PROMPT,
    build_prompt,
    checkpoint_exists,
    generate_explanation,
    load_checkpoint,
    save_checkpoint,
)

c:\Users\vimal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
# Set True to re-run even if explanations.json already exists
FORCE_RERUN = False

## 3. Checkpoint check

In [3]:
if checkpoint_exists(EXPLANATIONS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{EXPLANATIONS_PATH}'.")
    print("    Set FORCE_RERUN = True to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(EXPLANATIONS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will generate explanations.")

No checkpoint found (or FORCE_RERUN=True). Will generate explanations.


## 4. Load Stage 2 output

In [4]:
if results is None:
    if not ONTOLOGY_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Ontology results not found at '{ONTOLOGY_RESULTS_PATH}'.\n"
            "Please run 02_ontology.ipynb first."
        )
    ontology_data = load_checkpoint(ONTOLOGY_RESULTS_PATH)
    print(f"Loaded {len(ontology_data)} texts from Stage 2.")

[Checkpoint] Loaded 1 records ← 'outputs\ontology_results.json'
Loaded 1 texts from Stage 2.


## 5. Load LLM

In [5]:
if results is None:
    llm_tokenizer, llm_model = load_llm()

[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


[Loader] LLM ready.



## 6. [Optional] Preview prompt for a single sample

Run this cell to inspect what the LLM will receive before running the full batch.

In [6]:
if results is None:
    sample = ontology_data[0]          # ← change index to preview a different text
    prompt = build_prompt(
        predicted_class=sample["predicted_class"],
        feature_data=sample["feature_data"],
        user_category=sample["user_category"],
    )
    print("── SYSTEM PROMPT ─────────────────────────────────────────")
    print(SYSTEM_PROMPT)
    print("\n── USER PROMPT ───────────────────────────────────────────")
    print(prompt)

── SYSTEM PROMPT ─────────────────────────────────────────
You are an explanation generator for a biomedical XAI system.

You do NOT classify text.
You do NOT add external medical knowledge.
You do NOT infer physiology, causation, or mechanisms.

Your ONLY role:
Convert the structured explanation plan (LIME tokens + ontology ancestors +
model prediction) into a clear natural-language explanation.

Rules:
1. Use ONLY the provided ontology triples and model prediction.
2. NEVER add biomedical facts that are not explicitly listed.
3. If a concept has no relation to the predicted class, state this neutrally.
4. Beginner → broad/simple language.
5. Expert → technical terms and ontology hierarchy words.
6. DO NOT write with bullet points. Produce one coherent paragraph.
7. Be concise and strictly grounded.

── USER PROMPT ───────────────────────────────────────────
### INPUT
text: [REDACTED FOR BREVITY]
Prediction: Cardiovascular diseases

Salient tokens identified by LIME:
['hypertension']


## 7. Generate explanations (full batch)

In [7]:
if results is None:
    results = []
    for i, item in enumerate(ontology_data):
        print(f"\nGenerating explanation {i + 1}/{len(ontology_data)} …")
        print(f"  Class    : {item['predicted_class']}")
        print(f"  User     : {item['user_category']}")
        print(f"  Features : {[f['feature_word'] for f in item['feature_data']]}")

        explanation = generate_explanation(
            predicted_class=item["predicted_class"],
            feature_data=item["feature_data"],
            user_category=item["user_category"],
            tokenizer=llm_tokenizer,
            model=llm_model,
        )

        print(f"  Output   : {explanation[:120]}…")
        results.append({**item, "explanation": explanation})

    print("\n✅ Explanation generation complete.")


Generating explanation 1/1 …
  Class    : Cardiovascular diseases
  User     : EXPERT
  Features : ['hypertension']
  Output   : assistant
The user's input is related to cardiovascular diseases, specifically mentioning hypertension as a relevant fac…

✅ Explanation generation complete.


## 8. Inspect explanations

In [8]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Class   : {r['predicted_class']}")
    print(f"User    : {r['user_category']}")
    print(f"\nExplanation:\n{r['explanation']}")
    print()


── Text 1 ──────────────────────────
Class   : Cardiovascular diseases
User    : EXPERT

Explanation:
assistant
The user's input is related to cardiovascular diseases, specifically mentioning hypertension as a relevant factor. The prediction indicates that the user has a high likelihood of having cardiovascular diseases. The LIME analysis highlights hypertension as a significant feature, which is part of the broader category of cardiovascular diseases. This means that individuals who have hypertension are more likely to develop other types of cardiovascular diseases such as arterial diseases and vascular diseases.



## 9. Save checkpoint

In [9]:
save_checkpoint(results, EXPLANATIONS_PATH)
print(f"\n➡️  Continue to notebook 04_analysis.ipynb")

[Checkpoint] Saved 1 records → 'outputs\explanations.json'

➡️  Continue to notebook 04_analysis.ipynb
